In [ ]:
import pandas as pd

movies = pd.read_csv("../data/raw/movies.csv")

print("Shape:", movies.shape)
movies.head()

Shape: (9742, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB


In [ ]:
unique_genres = set()

for genre_list in movies["genres"]:
    unique_genres.update(genre_list.split("|"))

print("Number of unique genres:", len(unique_genres))
print(sorted(unique_genres))

Number of unique genres: 20
['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [ ]:
(movies["genres"] == "(no genres listed)").sum()

np.int64(34)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(token_pattern=r"[^|]+")

tfidf_matrix = tfidf.fit_transform(movies["genres"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (9742, 20)


In [ ]:
feature_names = tfidf.get_feature_names_out()

print(feature_names)

['(no genres listed)' 'action' 'adventure' 'animation' 'children' 'comedy'
 'crime' 'documentary' 'drama' 'fantasy' 'film-noir' 'horror' 'imax'
 'musical' 'mystery' 'romance' 'sci-fi' 'thriller' 'war' 'western']


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:", cosine_sim.shape)

Similarity Matrix Shape: (9742, 9742)


In [ ]:
indices = pd.Series(movies.index, index=movies["title"]).drop_duplicates()

print(indices.head())

title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64


In [ ]:
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return movies["title"].iloc[movie_indices]

In [ ]:
get_recommendations("Toy Story (1995)")

1706                                          Antz (1998)
2355                                   Toy Story 2 (1999)
2809       Adventures of Rocky and Bullwinkle, The (2000)
3000                     Emperor's New Groove, The (2000)
3568                                Monsters, Inc. (2001)
6194                                     Wild, The (2006)
6486                               Shrek the Third (2007)
6948                       Tale of Despereaux, The (2008)
7760    Asterix and the Vikings (Astérix et les Viking...
8219                                         Turbo (2013)
Name: title, dtype: str

In [ ]:
tags = pd.read_csv("../data/raw/tags.csv")

print("Shape:", tags.shape)
tags.head()

Shape: (3683, 4)


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [ ]:
print("Unique movies with tags:", tags["movieId"].nunique())

Unique movies with tags: 1572


In [ ]:
movie_tag_counts = tags.groupby("movieId")["tag"].count()

print(movie_tag_counts.describe())

count    1572.000000
mean        2.342875
std         5.562342
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       181.000000
Name: tag, dtype: float64


In [ ]:
movie_tags = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

movie_tags.head()

,movieId,tag
0,1,pixar pixar fun
1,2,fantasy magic board game Robin Williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake


In [ ]:
movies_with_tags = movies.merge(
    movie_tags,
    on="movieId",
    how="left"
)

movies_with_tags.head()

,movieId,title,genres,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game Robin Williams game
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,NaN
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake


In [ ]:
movies_with_tags["tag"] = movies_with_tags["tag"].fillna("")

movies_with_tags["features"] = (
    movies_with_tags["genres"].str.replace("|", " ", regex=False)
    + " "
    + movies_with_tags["tag"]
)

movies_with_tags[
    ["title", "features"]
].head()

,title,features
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy pi...
1,Jumanji (1995),Adventure Children Fantasy fantasy magic board...
2,Grumpier Old Men (1995),Comedy Romance moldy old
3,Waiting to Exhale (1995),Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy pregnancy remake


In [ ]:
movie_tags = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(sorted(set(x.astype(str)))))
    .reset_index()
)

movie_tags.head()

,movieId,tag
0,1,fun pixar
1,2,Robin Williams fantasy game magic board game
2,3,moldy old
3,5,pregnancy remake
4,7,remake


In [ ]:
movies_with_tags = movies.merge(
    movie_tags,
    on="movieId",
    how="left"
)

movies_with_tags["tag"] = movies_with_tags["tag"].fillna("")

movies_with_tags["features"] = (
    movies_with_tags["genres"].str.replace("|", " ", regex=False)
    + " "
    + movies_with_tags["tag"]
)

movies_with_tags[["title", "features"]].head()

,title,features
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy fu...
1,Jumanji (1995),Adventure Children Fantasy Robin Williams fant...
2,Grumpier Old Men (1995),Comedy Romance moldy old
3,Waiting to Exhale (1995),Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy pregnancy remake


In [ ]:
movies_with_tags.loc[
    movies_with_tags["title"] == "Toy Story (1995)",
    ["title", "features"]
]

,title,features
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy fu...


In [ ]:
movies_with_tags.loc[
    movies_with_tags["title"] == "Guardians of the Galaxy 2 (2017)",
    ["title", "features"]
]

,title,features
8695,Guardians of the Galaxy 2 (2017),Action Adventure Sci-Fi fun


In [ ]:
(tags["tag"].str.lower() == "fun").sum()

np.int64(5)